** SILVER LAYER — clean, typed, production-ready records**

- What happens here (Step 1 — basics only):
- Read from Bronze (batch read — Silver always reads from Bronze,
- not from the raw files directly)
- Cast columns to correct types (Bronze stores everything as string
- unless cloudFiles.inferColumnTypes was true — we still cast
- explicitly here for safety and documentation)
- Drop the _rescued_data column for Silver (we keep it in Bronze)
- Write to Silver as a managed Delta table
- overwrite for Step 1 (we switch to MERGE in Step 3 - Idempotency)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType, DoubleType, DateType, TimestampType, StructType, StructField, StringType
from delta.tables import DeltaTable
import uuid
from datetime import datetime

In [0]:
# Business lateness window:
# Orders arriving more than this many days after their order_date
# are flagged as late. Those beyond BEYOND_WINDOW_DAYS are flagged
# separately as needing investigation (e.g. data quality issue).
LATENESS_WINDOW_DAYS = 5
BEYOND_WINDOW_DAYS   = 30

# 4. SILVER — incremental read + dedup + MERGE+ late arriving file

- Step A: fetch last watermark from control table
- Step B: read ONLY Bronze rows newer than the watermark
- Step C: dedup the incoming batch (same as Step 2)
- Step D: MERGE into Silver (same as Step 3)
- Step E: save the new watermark (highest _ingest_timestamp in this batch)

In [0]:
bronze_table=spark.table("file_based.ecommerce.bronze_layer")

In [0]:
bronze_table.printSchema()

In [0]:
silver_table="file_based.ecommerce.Silver_table"



In [0]:
def get_last_watermark(table_name_t):
    control_table="file_based.ecommerce.pipeline_control"
    control_table_exists = spark.catalog.tableExists(control_table)
    if not control_table_exists:
        return datetime(1970,1,1)
    else:
        row=(
            spark.table(control_table).filter(F.col("table_name") == table_name_t).select("last_watermark").collect()
        )
    if not row:
        return datetime(1970, 1, 1)
    
    return row[0]["last_watermark"]


    

In [0]:
def save_watermark(table_name,new_watermark,batch_id):
    control_table="file_based.ecommerce.pipeline_control"

    new_row=spark.createDataFrame(
        [(table_name,new_watermark,batch_id,datetime.now())],
        schema=StructType([
            StructField("table_name",StringType(),False),
            StructField("last_watermark",TimestampType(),False),
            StructField("last_batch_id",StringType(),False),
            StructField("last_run_id",TimestampType(),False),
        ])
    )

    control_exists=spark.catalog.tableExists(control_table)
    if not control_exists:
        new_row.write.format("delta").saveAsTable(control_table)
    else:
        (
            DeltaTable.forName(spark, control_table).alias("ctrl")
            .merge(
                new_row.alias("incoming"),
                "ctrl.table_name = incoming.table_name"
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )
    print(f"Watermark saved → {new_watermark} for '{table_name}'")


In [0]:
%sql

select * from file_based.ecommerce.bronze_layer

In [0]:
EXPECTED_COLUMNS = {
    "order_id", "order_line_id", "customer_id", "order_date",
    "order_status", "product_id", "product_category",
    "quantity", "unit_price", "order_total", "currency",
}

In [0]:
SILVER_SCHEMA_LOG   = "file_based.ecommerce.silver_schema_log"

In [0]:
from datetime import datetime
from pyspark.sql.types import (
    StructType, StructField,
    StringType, TimestampType
)

def check_schema_changes(batch_id):

    # Incoming Bronze schema
    incoming_cols = {
        f.name: f.dataType
        for f in spark.table("file_based.ecommerce.bronze_layer").schema.fields
        if not f.name.startswith("_")
    }

    incoming_column_names = set(incoming_cols.keys())

    new_columns = incoming_column_names - EXPECTED_COLUMNS
    missing_columns = EXPECTED_COLUMNS - incoming_column_names

    schema_log_rows = []
    has_breaking = False

    # ------------------------------------------------------------------
    # Additive schema changes
    # ------------------------------------------------------------------
    if new_columns:
        print(f"⚠️ Additive schema change detected: {new_columns}")

        for col in new_columns:
            schema_log_rows.append((
                batch_id,
                "additive",
                col,
                str(incoming_cols[col]),
                None,
                datetime.now(),
                "Logged - Silver will absorb via mergeSchema"
            ))

    # ------------------------------------------------------------------
    # Breaking schema changes (datatype changes)
    # ------------------------------------------------------------------
    # common_columns = EXPECTED_COLUMNS.intersection(incoming_column_names)

    # for col in common_columns:

    #     expected_type = EXPECTED_COLUMNS[col]
    #     incoming_type = incoming_cols[col]

    #     if expected_type != incoming_type:

    #         has_breaking = True

    #         print(
    #             f"❌ Breaking schema change detected: "
    #             f"'{col}' changed from {expected_type} to {incoming_type}"
    #         )

    #         schema_log_rows.append((
    #             batch_id,
    #             "breaking",
    #             col,
    #             str(incoming_type),
    #             str(expected_type),
    #             datetime.now(),
    #             "Pipeline stopped"
    #         ))

    # ------------------------------------------------------------------
    # Missing columns
    # ------------------------------------------------------------------
    if missing_columns:
        print(f"ℹ️ Expected columns not present in this batch: {missing_columns}")
        print("   Missing columns will be validated during DQ checks.")

    # ------------------------------------------------------------------
    # Write schema log
    # ------------------------------------------------------------------
    if schema_log_rows:

        log_schema = StructType([
            StructField("batch_id", StringType(), False),
            StructField("change_type", StringType(), False),
            StructField("column_name", StringType(), False),
            StructField("new_type", StringType(), True),
            StructField("previous_type", StringType(), True),
            StructField("detected_at", TimestampType(), False),
            StructField("action_taken", StringType(), False),
        ])

        log_df = spark.createDataFrame(schema_log_rows, schema=log_schema)

        if not spark.catalog.tableExists(SILVER_SCHEMA_LOG):
            log_df.write.format("delta").saveAsTable(SILVER_SCHEMA_LOG)
        else:
            log_df.write.format("delta").mode("append").saveAsTable(SILVER_SCHEMA_LOG)

    return has_breaking

In [0]:

#one batch_id  for this entire pipeline run 
batch_id=str(uuid.uuid4())
print(f"Pipeline run batch_id: {batch_id}")

#step1: fetch last watermark from control table
last_watermark=get_last_watermark(silver_table)
print(f"last watermark : {last_watermark}")

# Run schema check before Silver processing
has_breaking_change = check_schema_changes(batch_id)
 
if has_breaking_change:
    raise Exception("Breaking schema change detected. Stopping pipeline. Check schema_change_log.")

# ── Step 2: incremental Bronze read — only new rows since last watermark
#    This is the key change vs Step 3. Instead of reading all of Bronze,
#    we filter to only rows ingested after the last successful run.

incremental_bronze=(
    spark.table("file_based.ecommerce.bronze_layer").filter(F.col("_ingest_timestamp")>F.lit(last_watermark).cast(TimestampType()))
)

incremental_count=incremental_bronze.count()
print(f"New Bronze rows to process: {incremental_count}")

if incremental_count == 0:
    print("No new data since last watermark. Silver is already up to date.")
    dbutils.notebook.exit("No new data to process.")




#deduplication
depud=Window.partitionBy("order_id","order_line_id").orderBy(F.col("_ingest_timestamp").desc())


incoming_df=(
    incremental_bronze.withColumn("order_date", F.col("order_date").cast(DateType()))
    .withColumn("quantity",     F.col("quantity").cast(IntegerType()))
    .withColumn("unit_price",   F.col("unit_price").cast(DoubleType()))
    .withColumn("order_total",  F.col("order_total").cast(DoubleType()))
    .drop("_rescued_data")
    .withColumn("_silver_processed_at", F.current_timestamp())
    .withColumn("_batch_id", F.lit(batch_id)) # for idempotency
    .withColumn("_row_num",F.row_number().over(depud)) # for deduplication
    .filter(F.col("_row_num")==1)
    .drop("_row_num")
    .withColumn("processing_date",F.to_date(F.col("_ingest_timestamp")))
    .withColumn("arrival_delay_days",
                F.datediff(F.col("processing_date"), F.col("order_date")))
    .withColumn("is_late_arrival",
                (F.col("arrival_delay_days") > 0) &
                (F.col("arrival_delay_days") <= LATENESS_WINDOW_DAYS))
    .withColumn("is_beyond_lateness_window",
                F.col("arrival_delay_days") > BEYOND_WINDOW_DAYS)

)
silver_table="file_based.ecommerce.Silver_table"
silver_table_exists = spark.catalog.tableExists(silver_table)

if not silver_table_exists:
    (
        incoming_df.write
        .format("delta")
        .saveAsTable(silver_table)
    )
    print(f"Silver table created on first run. Rows: {incoming_df.count()}")
else:
    silver_table_delta = DeltaTable.forName(spark, silver_table)

    (
        silver_table_delta.alias("sd")
        .merge(
            incoming_df.alias("df"),
            "sd.order_id = df.order_id AND sd.order_line_id = df.order_line_id",
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

silver_count = spark.table(silver_table).count()
print(f"Silver MERGE complete. Total rows in Silver: {silver_count}")

# ── Step E: save new watermark — highest _ingest_timestamp in this batch
new_watermark=(
    incremental_bronze.agg(F.max("_ingest_timestamp")).collect()[0][0]
)

save_watermark(silver_table,new_watermark,batch_id)

In [0]:
%sql
select * from file_based.ecommerce.Silver_table

In [0]:
late_arrival_dates = [
    row["order_date"]
    for row in (
        incoming_df
        .filter(
            F.col("is_late_arrival") | F.col("is_beyond_lateness_window")
        )
        .select("order_date")
        .distinct()
        .collect()
    )
]

In [0]:
print(f"\nLate arrival order_dates affected: {late_arrival_dates}")

In [0]:
silver_df = spark.table("file_based.ecommerce.silver_table")
 
on_time_count   = silver_df.filter(F.col("arrival_delay_days") == 0).count()
late_count      = silver_df.filter(F.col("is_late_arrival") == True).count()
beyond_count    = silver_df.filter(F.col("is_beyond_lateness_window") == True).count()

In [0]:
print("=" * 55)
print("Late Arrival Verification")
print("=" * 55)
print(f"  On-time records             : {on_time_count}")
print(f"  Late (within window)        : {late_count}")
print(f"  Beyond lateness window      : {beyond_count}")
print(f"  Lateness window             : {LATENESS_WINDOW_DAYS} days")
print()

In [0]:
# Show a sample of late records so you can confirm the logic is right
print("Sample late arriving records:")
display(
    silver_df
    .filter(F.col("is_late_arrival") == True)
    .select("order_id", "order_date", "processing_date",
            "arrival_delay_days", "is_late_arrival",
            "is_beyond_lateness_window")
    .orderBy(F.col("arrival_delay_days").desc())
    .limit(10)
)

In [0]:
print("=" * 55)
print("Incremental Load Verification")
print("=" * 55)
print(f"  Bronze total rows         : {spark.table("file_based.ecommerce.bronze_layer").count()}")
print(f"  Rows read this run        : {incremental_count}  ← only new rows")
print(f"  Silver total rows         : {spark.table("file_based.ecommerce.silver_table").count()}")
print()
print("Control table (watermarks):")
display(spark.table("file_based.ecommerce.pipeline_control"))

In [0]:
silver_count = spark.table(silver_table).count()
print(f"Silver MERGE complete. Rows: {silver_count}")

In [0]:
display(spark.table("file_based.ecommerce.Silver_Table"))

In [0]:
bronze_count = spark.table("file_based.ecommerce.bronze_layer").count()
silver_count = spark.table(silver_table).count()

In [0]:
# Duplicate keys check
remaining_dupes = (
    spark.table(silver_table)
    .groupBy("order_id", "order_line_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

In [0]:
# Distinct batch IDs — tells you how many pipeline runs touched Silver
batch_ids = (
    spark.table(silver_table)
    .select("_batch_id")
    .distinct()
    .count()
)

In [0]:
print("=" * 55)
print("Idempotency Verification")
print("=" * 55)
print(f"  Bronze rows              : {bronze_count}")
print(f"  Silver rows              : {silver_count}")
print(f"  Duplicate keys in Silver : {remaining_dupes}  (must be 0)")
print(f"  Distinct batch IDs       : {batch_ids}")
print()
 
# Re-run cell 4 (MERGE) and then re-run this cell.
# Silver count must stay the same — that is idempotency.
print("→ Re-run the MERGE cell now, then re-run this cell.")
print("  Silver count must remain:", silver_count)
 
# COMMAND ----------
 
# Show the _batch_id column — useful for understanding reruns
display(
    spark.table(silver_table)
    .select("order_id", "order_line_id", "order_date", "order_total", "_batch_id", "_silver_processed_at")
    .orderBy("order_id")
    .limit(10)
)